# Feature Engineering for ML Targets (Aligned with KPI Logic)

This notebook creates ML-ready target variables.

All formulas replicate the KPI logic implemented in `kpi_engine.py` to ensure consistency between:
- Dashboard metrics
- Backend calculations
- ML training targets

The purpose is traceability and reproducibility.

If KPI formulas change in the future, this notebook must be updated accordingly.

This prevents inconsistencies between model predictions and dashboard insights.

## Methodological Explanation

### 1) Dependency Risk
- Built from C1-C4 items.
- C3 and C4 are inverted using: `8 - value`.
- Final score is the mean of the 4 transformed items.
- Categorization (aligned with backend KPI thresholds):
  - `high` if score > 5.5
  - `medium` if score is in (3.5, 5.5]
  - `low` if score <= 3.5
- ML targets created:
  - `dependency_level` (multiclass)
  - `dependency_risk_high` (binary: high vs rest)

Justification:
- This reflects behavioral dependency patterns.
- Thresholds are defined in backend KPI logic and are kept unchanged for strict alignment.

### 2) Motivation Score
- Computed as mean of M1-M4.
- Continuous target variable.
- Optional delta against neutral Likert midpoint (`4`): `delta_motivation_score`.

Justification:
- The midpoint represents neutral motivation.
- This supports regression-based modeling.

### 3) Self-Efficacy Score
- Computed as mean of AE1-AE4.
- Used as an engineered feature and as part of dropout proxy logic.

### 4) Dropout Risk (Proxy)
- Binary = 1 if:
  - `dependency_risk > 5.5`
  - OR `motivation_score <= 3.5`
  - OR `self_efficacy_score <= 3.5`
- Otherwise 0.

Justification:
- This is a composite disengagement risk proxy built from available cross-sectional signals.

### 5) Composite ROI Score
- Mean of block means:
  - `AT_mean` (AT1-AT4)
  - `D_mean` (D1-D4)
  - `E_mean` (E1-E4)
- Final score: `composite_roi_score = mean(AT_mean, D_mean, E_mean)`.

Why proxies are necessary:
- The dataset is cross-sectional.
- No time-series baseline exists.
- Therefore, interpretable composite indicators are constructed for ML usage.

In [2]:
# Load standard library and allowed dependencies only.
import os
import pandas as pd
import numpy as np


def _find_project_root(start_path: str) -> str:
    """Find project root by walking up until .env and backend folder are found."""
    current = os.path.abspath(start_path)
    while True:
        env_candidate = os.path.join(current, '.env')
        backend_candidate = os.path.join(current, 'backend')
        if os.path.isfile(env_candidate) and os.path.isdir(backend_candidate):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise ValueError('Could not locate project root containing .env and backend directory.')


def _read_env_value(env_file: str, key: str) -> str:
    """Read a single key from .env file without external dependencies."""
    if not os.path.isfile(env_file):
        return ''
    with open(env_file, 'r', encoding='utf-8') as f:
        for line in f:
            raw = line.strip()
            if not raw or raw.startswith('#') or '=' not in raw:
                continue
            k, v = raw.split('=', 1)
            if k.strip() == key:
                return v.strip().strip('"').strip("'")
    return ''


project_root = _find_project_root(os.getcwd())

# Read data path from environment variable as requested.
data_path = os.getenv('DATA_PATH')

# Fallback to .env when DATA_PATH is not injected into the notebook process.
if not data_path:
    env_path = os.path.join(project_root, '.env')
    data_path = _read_env_value(env_path, 'DATA_PATH')

# Validate that DATA_PATH is provided.
if not data_path:
    raise ValueError('DATA_PATH environment variable is not set (and not found in .env).')

# Resolve relative DATA_PATH from project root.
if not os.path.isabs(data_path):
    data_path = os.path.join(project_root, data_path)

# Validate that the file exists before loading.
if not os.path.exists(data_path):
    raise FileNotFoundError(f'Dataset not found at: {data_path}')

# Load file based on extension while using pandas only.
if data_path.lower().endswith('.csv'):
    # Dataset in this project uses semicolon delimiter.
    df = pd.read_csv(data_path, sep=';')
elif data_path.lower().endswith('.xls') or data_path.lower().endswith('.xlsx'):
    df = pd.read_excel(data_path)
else:
    raise ValueError(f'Unsupported file extension for DATA_PATH: {data_path}')

# Print shape for verification.
print(f'Dataset loaded from: {data_path}')
print(f'Dataset shape: {df.shape}')

Dataset loaded from: /workspaces/EvaluAI/backend/data/raw/EIPIA_FO_dataset_100_personas.xls
Dataset shape: (100, 41)


In [3]:
# Compute dependency risk using the exact KPI logic from kpi_engine.py.
# Step 1: invert C3 and C4 exactly as defined: 8 - value.
df['C3_inv'] = 8 - pd.to_numeric(df['C3_pensamiento_critico'], errors='coerce')
df['C4_inv'] = 8 - pd.to_numeric(df['C4_reflexiono_calidad'], errors='coerce')

# Step 2: compute dependency_risk as mean of C1, C2, C3_inv, C4_inv.
df['dependency_risk'] = pd.concat([
    pd.to_numeric(df['C1_confio_sin_verificar'], errors='coerce'),
    pd.to_numeric(df['C2_dificil_sin_ia'], errors='coerce'),
    df['C3_inv'],
    df['C4_inv']
], axis=1).mean(axis=1)

# Step 3: categorize dependency level using exact backend thresholds.
df['dependency_level'] = np.where(
    df['dependency_risk'] > 5.5,
    'high',
    np.where(df['dependency_risk'] > 3.5, 'medium', 'low')
)

# Step 4: create binary high-risk target.
df['dependency_risk_high'] = (df['dependency_level'] == 'high').astype(int)

# Print multiclass and binary distributions.
print('dependency_level distribution:')
print(df['dependency_level'].value_counts(dropna=False))
print('\ndependency_risk_high distribution:')
print(df['dependency_risk_high'].value_counts(dropna=False))

dependency_level distribution:
dependency_level
medium    64
low       31
high       5
Name: count, dtype: int64

dependency_risk_high distribution:
dependency_risk_high
0    95
1     5
Name: count, dtype: int64


In [4]:
# Compute motivation score from M1-M4 as a simple mean.
df['motivation_score'] = df[[
    'M1_estimulante',
    'M2_aumenta_interes',
    'M3_aporta_valor',
    'M4_mayor_esfuerzo'
]].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# Compute delta versus neutral midpoint of Likert scale (4).
df['delta_motivation_score'] = df['motivation_score'] - 4

# Create binary improved motivation flag.
df['motivation_improved'] = (df['delta_motivation_score'] > 0).astype(int)

# Print summary statistics and binary distribution.
print('motivation_score summary:')
print(df['motivation_score'].describe())
print('\ndelta_motivation_score summary:')
print(df['delta_motivation_score'].describe())
print('\nmotivation_improved distribution:')
print(df['motivation_improved'].value_counts(dropna=False))

motivation_score summary:
count    100.000000
mean       3.992500
std        0.980527
min        1.750000
25%        3.250000
50%        4.000000
75%        4.562500
max        6.500000
Name: motivation_score, dtype: float64

delta_motivation_score summary:
count    100.000000
mean      -0.007500
std        0.980527
min       -2.250000
25%       -0.750000
50%        0.000000
75%        0.562500
max        2.500000
Name: delta_motivation_score, dtype: float64

motivation_improved distribution:
motivation_improved
0    56
1    44
Name: count, dtype: int64


In [5]:
# Compute self-efficacy score from AE1-AE4 as a simple mean.
df['self_efficacy_score'] = df[[
    'AE1_resolver_problemas',
    'AE2_confianza_digital',
    'AE3_uso_eficaz',
    'AE4_seguridad_aplicacion'
]].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# Print summary statistics for self-efficacy.
print('self_efficacy_score summary:')
print(df['self_efficacy_score'].describe())

self_efficacy_score summary:
count    100.000000
mean       4.007500
std        1.039895
min        1.000000
25%        3.250000
50%        4.000000
75%        4.750000
max        6.750000
Name: self_efficacy_score, dtype: float64


In [6]:
# Compute dropout risk proxy using the specified OR rule.
# dropout_risk = 1 if dependency_risk > 5.5 OR motivation_score <= 3.5 OR self_efficacy_score <= 3.5.
df['dropout_risk'] = ((df['dependency_risk'] > 5.5) |
                      (df['motivation_score'] <= 3.5) |
                      (df['self_efficacy_score'] <= 3.5)).astype(int)

# Print class distribution for dropout_risk.
print('dropout_risk distribution:')
print(df['dropout_risk'].value_counts(dropna=False))

dropout_risk distribution:
dropout_risk
1    63
0    37
Name: count, dtype: int64


In [7]:
# Compute block means for AT, D, and E groups.
df['AT_mean'] = df[[
    'AT1_rendimiento',
    'AT2_facilita_aprendizaje',
    'AT3_facilidad_uso',
    'AT4_integracion_positiva'
]].apply(pd.to_numeric, errors='coerce').mean(axis=1)

df['D_mean'] = df[[
    'D1_mejora_competencias',
    'D2_preparado_retos',
    'D3_amplia_habilidades',
    'D4_aprendizaje_autonomo'
]].apply(pd.to_numeric, errors='coerce').mean(axis=1)

df['E_mean'] = df[[
    'E1_adaptacion',
    'E2_feedback_util',
    'E3_aplicable_trabajo',
    'E4_aprendizaje_ritmo'
]].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# Compute composite ROI score as mean of AT_mean, D_mean, E_mean.
df['composite_roi_score'] = df[['AT_mean', 'D_mean', 'E_mean']].mean(axis=1)

# Print summary statistics for composite ROI score.
print('composite_roi_score summary:')
print(df['composite_roi_score'].describe())

composite_roi_score summary:
count    100.000000
mean       4.015000
std        0.556234
min        3.000000
25%        3.583333
50%        4.000000
75%        4.333333
max        5.583333
Name: composite_roi_score, dtype: float64


In [8]:
# Save processed dataset for ML usage.
# Resolve project root and read output directory from environment for maintainability.

def _find_project_root(start_path: str) -> str:
    """Find project root by walking up until backend/data/processed exists."""
    current = os.path.abspath(start_path)
    while True:
        candidate = os.path.join(current, 'backend', 'data', 'processed')
        if os.path.isdir(candidate):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise ValueError(
        'Could not locate project root containing backend/data/processed.'
    )

# Optional validation: DATA_PATH should reference backend/data/raw.
normalized_data_path = os.path.abspath(data_path).replace('\\', '/')
if '/backend/data/raw/' not in normalized_data_path:
    raise ValueError(
        'Invalid DATA_PATH structure. Expected pattern like: <repo>/backend/data/raw/<file>'
    )

project_root = _find_project_root(os.getcwd())

# Read output directory from environment (set in .env), with safe default.
engineered_data_dir = os.getenv('ENGINEERED_DATA_DIR', 'backend/data/processed')

# If relative, resolve from project root; if absolute, use as-is.
if os.path.isabs(engineered_data_dir):
    output_dir = engineered_data_dir
else:
    output_dir = os.path.join(project_root, engineered_data_dir)

output_path = os.path.join(output_dir, 'survey_engineered.csv')

# Ensure output directory exists and save CSV.
os.makedirs(output_dir, exist_ok=True)
df.to_csv(output_path, index=False)

# Print confirmation message.
print(f'Processed dataset saved to: {output_path}')
print(f'Rows: {len(df)}, Columns: {len(df.columns)}')

Processed dataset saved to: /workspaces/EvaluAI/backend/data/processed/survey_engineered.csv
Rows: 100, Columns: 55


## Final Summary

This notebook creates the following engineered ML targets:
- `dependency_level` (multiclass)
- `dependency_risk_high` (binary)
- `motivation_score` (continuous)
- `delta_motivation_score` (continuous)
- `motivation_improved` (binary)
- `self_efficacy_score` (continuous)
- `dropout_risk` (binary proxy)
- `composite_roi_score` (continuous)

Class distributions are printed in previous cells for traceability.

Suggested model family by target type:
- **RandomForestClassifier**:
  - `dependency_risk_high`
  - `dependency_level`
  - `motivation_improved`
  - `dropout_risk`
- **RandomForestRegressor**:
  - `motivation_score`
  - `delta_motivation_score`
  - `self_efficacy_score`
  - `composite_roi_score`

Important modeling notes:
- `dependency_risk_high` shows severe class imbalance (around 5% high-risk).
- Prefer `dependency_level` multiclass modeling over high-only binary target when possible.
- `dropout_risk` typically has more acceptable balance than `dependency_risk_high`.
- Continuous targets are generally suitable for regression workflows due to broader value spread.

## 📊 Estado real de los targets (análisis profesional)

**Nota:** Este análisis se ha realizado utilizando únicamente los datos disponibles para la POC (100 empleados).  
Los resultados deben interpretarse teniendo en cuenta esta limitación de tamaño muestral.

---

### 🔴 dependency_level

**Distribución:**

- medium → 64%  
- low → 31%  
- high → 5%

Esto confirma que:

- La clase **high** es extremadamente minoritaria.
- El modelado multiclass es viable, pero:
  - El modelo tendrá dificultad real detectando la clase “high”.
  - El F1-score para esa clase será bajo.
  - La accuracy global puede resultar engañosa.

👉 Para demo es válido.  
👉 Para producción sería necesario ampliar el dataset.

---

### 🔴 dependency_risk_high (binario puro)

**Distribución:**

- 0 → 95  
- 1 → 5  

Esto **no es entrenable de forma robusta** con solo 100 muestras.

Incluso usando:

```python
class_weight="balanced"
```

El modelo no será estable ni generalizable.

👉 Recomendación firme: **no entrenar este modelo como principal.**

---

### 🟢 motivation_score

**Distribución:**

- Media ≈ 4

- Desviación estándar ≈ 0.98

- Rango amplio (1.75 – 6.5)

Esto indica:

- Buena dispersión.

- Distribución equilibrada.

- Adecuado para modelado de regresión.

👉 Excelente target para `RandomForestRegressor`.
👉 Este modelo puede ofrecer métricas coherentes.

---

### 🟢 dropout_risk

**Distribución:**

- 1 → 63%

- 0 → 37%

Esto implica:

- Balance aceptable.

- Modelo entrenable.

- Métricas interpretables.

- Caso de uso claro para RRHH.

👉 Este es un modelo sólido para la demo.

---

### 🟢 composite_roi_score

**Distribución:**

- Media ≈ 4.01

- Desviación estándar ≈ 0.55

Aunque presenta menor dispersión que otros targets, sigue siendo válida.

👉 Adecuado para regresión.
👉 Especialmente útil para insights ejecutivos y toma de decisiones estratégicas.

